# Day 30 — Data Engineering Integration

## Objectives
- Validate stock-price database
- Validate economic-data database
- Test the reusable economic analytics module
- Verify paper-trading database structure
- Produce a data-engineering health report

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

from economic_analytics import get_economic_snapshot

PRICE_DB = Path("hedge_fund.db")
TRADING_DB = Path("paper_trading.db")

assert PRICE_DB.exists(), "hedge_fund.db is missing."
assert TRADING_DB.exists(), "paper_trading.db is missing."

print("PASS: Both databases exist.")

PASS: Both databases exist.


In [2]:
def inspect_database(database_path):
    with sqlite3.connect(database_path) as conn:
        tables = pd.read_sql_query("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
              AND name NOT LIKE 'sqlite_%'
            ORDER BY name
        """, conn)

        results = []

        for table_name in tables["name"]:
            count = conn.execute(
                f'SELECT COUNT(*) FROM "{table_name}"'
            ).fetchone()[0]

            results.append({
                "database": database_path.name,
                "table": table_name,
                "records": count
            })

    return pd.DataFrame(results)


price_tables = inspect_database(PRICE_DB)
trading_tables = inspect_database(TRADING_DB)

database_report = pd.concat(
    [price_tables, trading_tables],
    ignore_index=True
)

display(database_report)

assert "daily_prices" in price_tables["table"].values
assert "economic_vintages" in price_tables["table"].values

for table in ["accounts", "positions", "trades"]:
    assert table in trading_tables["table"].values, (
        f"Missing paper-trading table: {table}"
    )

print("PASS: Required database tables exist.")

,database,table,records
0,hedge_fund.db,daily_prices,1250
1,hedge_fund.db,derived_economic_indicators,172
2,hedge_fund.db,economic_data,464
3,hedge_fund.db,economic_vintage_snapshots,105
4,hedge_fund.db,economic_vintages,170
5,hedge_fund.db,portfolio_holdings,5
6,hedge_fund.db,portfolio_weights,5
7,paper_trading.db,accounts,1
8,paper_trading.db,equity_snapshots,1
9,paper_trading.db,positions,1


PASS: Required database tables exist.


In [3]:
with sqlite3.connect(PRICE_DB) as conn:
    economic_summary = pd.read_sql_query("""
        SELECT
            indicator,
            COUNT(*) AS records,
            MIN(observation_date) AS first_observation,
            MAX(observation_date) AS latest_observation,
            MAX(available_date) AS latest_available_date
        FROM economic_vintages
        GROUP BY indicator
        ORDER BY indicator
    """, conn)

display(economic_summary)

required_indicators = {
    "CPI",
    "UNEMPLOYMENT",
    "REAL_GDP",
    "FED_RATE"
}

assert required_indicators.issubset(
    set(economic_summary["indicator"])
)

assert (economic_summary["records"] > 0).all()

print("PASS: All four economic indicators contain data.")

,indicator,records,first_observation,latest_observation,latest_available_date
0,CPI,66,2024-01-01,2026-08-01,2026-09-11
1,FED_RATE,32,2024-01-01,2026-08-01,2026-09-01
2,REAL_GDP,34,2024-01-01,2026-04-01,2026-08-26
3,UNEMPLOYMENT,38,2024-01-01,2026-08-01,2026-09-04


PASS: All four economic indicators contain data.


In [4]:
with sqlite3.connect(PRICE_DB) as conn:
    stock_summary = pd.read_sql_query("""
        SELECT
            ticker,
            COUNT(*) AS trading_days,
            MIN(date) AS first_date,
            MAX(date) AS latest_date,
            SUM(
                CASE
                    WHEN close_price IS NULL OR close_price <= 0
                    THEN 1 ELSE 0
                END
            ) AS invalid_prices
        FROM daily_prices
        GROUP BY ticker
        ORDER BY ticker
    """, conn)

display(stock_summary)

expected_tickers = {"AAPL", "BLK", "GS", "JPM", "MSFT"}

assert expected_tickers.issubset(
    set(stock_summary["ticker"])
)

assert (stock_summary["invalid_prices"] == 0).all()

print("PASS: Historical stock-price data validated.")

,ticker,trading_days,first_date,latest_date,invalid_prices
0,AAPL,250,2025-09-23,2026-09-21,0
1,BLK,250,2025-09-23,2026-09-21,0
2,GS,250,2025-09-23,2026-09-21,0
3,JPM,250,2025-09-23,2026-09-21,0
4,MSFT,250,2025-09-23,2026-09-21,0


PASS: Historical stock-price data validated.


In [5]:
with sqlite3.connect(TRADING_DB) as conn:
    account = pd.read_sql_query(
        "SELECT * FROM accounts",
        conn
    )

    trades = pd.read_sql_query(
        "SELECT * FROM trades",
        conn
    )

    positions = pd.read_sql_query(
        "SELECT * FROM positions",
        conn
    )

print("Paper-trading accounts:")
display(account)

print("Trade history:")
display(trades)

print("Current positions:")
display(positions)

assert not account.empty
assert (account["cash_balance"] >= 0).all()

print("PASS: Paper-trading database validated.")

Paper-trading accounts:


,account_id,starting_capital,cash_balance
0,1,100000.0,99100.0


Trade history:


,trade_id,timestamp,ticker,side,quantity,execution_price,fees,realized_pnl
0,1,2026-09-23T22:50:08.319719+00:00,AAPL,BUY,10.0,200.0,0.0,0.0
1,2,2026-09-23T22:53:32.624972+00:00,AAPL,SELL,5.0,220.0,0.0,100.0


Current positions:


,ticker,quantity,average_cost
0,AAPL,5.0,200.0


PASS: Paper-trading database validated.
